# 🛡️ Agent Governance Toolkit — Getting Started

This notebook walks through the core governance primitives:
1. **Policy Engine** — Define and evaluate governance rules
2. **Trust Identity** — Cryptographic agent credentials
3. **Audit Events** — Structured governance logging
4. **Governance Verification** — OWASP ASI 2026 compliance check

## 1. Policy Engine

The `PolicyEngine` evaluates every agent action against governance rules.

In [ ]:
from agent_os.policy import PolicyEngine, CapabilityModel

# Define capabilities — what the agent is allowed to do
capabilities = CapabilityModel(
    allowed_tools=["web_search", "read_file", "calculator"],
    blocked_tools=["execute_code", "delete_file"],
    blocked_patterns=[r"\b\d{3}-\d{2}-\d{4}\b"],  # Block SSN patterns
    require_human_approval=False,
    max_tool_calls=10,
)

engine = PolicyEngine(capabilities=capabilities)
print("✅ PolicyEngine initialized")
print(f"   Allowed tools: {capabilities.allowed_tools}")
print(f"   Blocked tools: {capabilities.blocked_tools}")

In [ ]:
# Evaluate an allowed action
result = engine.evaluate(action="web_search", input_text="latest AI news")
print(f"web_search → allowed={result.allowed}, reason='{result.reason}'")

# Evaluate a blocked action
result = engine.evaluate(action="delete_file", input_text="/etc/passwd")
print(f"delete_file → allowed={result.allowed}, reason='{result.reason}'")

# Evaluate with PII (SSN pattern blocked)
result = engine.evaluate(action="web_search", input_text="Look up 123-45-6789")
print(f"SSN input → allowed={result.allowed}, reason='{result.reason}'")

## 2. Trust Identity

Every agent gets a cryptographic identity with a DID (Decentralized Identifier).

In [ ]:
from agentmesh.identity import AgentIdentity

# Create an agent identity
identity = AgentIdentity.create("research-agent")
print(f"Agent DID: {identity.did}")
print(f"Public key: {identity.public_key[:16]}...")

# Sign a message
message = b"I approve this action"
signature = identity.sign(message)
print(f"Signature: {signature[:16]}...")

# Verify the signature
verified = identity.verify(message, signature)
print(f"Verified: {verified}")

## 3. Governance Verification

Check OWASP ASI 2026 coverage and generate attestation badges.

In [ ]:
from agent_compliance.verify import GovernanceVerifier

verifier = GovernanceVerifier()
attestation = verifier.verify()

print(attestation.summary())
print()
print(f"Controls: {attestation.controls_passed}/{attestation.controls_total}")
print(f"Badge: {attestation.badge_markdown()}")

## 4. Module Integrity

Verify that no governance modules have been tampered with.

In [ ]:
from agent_compliance.integrity import IntegrityVerifier

verifier = IntegrityVerifier()
report = verifier.verify()

print(f"Modules checked: {report.total_modules}")
print(f"All intact: {report.all_passed}")
for m in report.modules[:5]:
    icon = '✅' if m.passed else '❌'
    print(f"  {icon} {m.name}")

## Next Steps

- **Framework integration**: See [QUICKSTART.md](../QUICKSTART.md) for LangChain/OpenAI wrapping
- **OWASP compliance**: See [docs/OWASP-COMPLIANCE.md](../docs/OWASP-COMPLIANCE.md)
- **Examples**: Explore [agent-governance-python/agent-os/examples/](../agent-governance-python/agent-os/examples/)
- **Contributing**: See [CONTRIBUTING.md](../CONTRIBUTING.md)